# 01 — Pipeline smoke test (tiny encoder)

End-to-end check that this repo runs: a tiny real encoder plus one epoch on a
small slice of `LocalLLaMA/typed-decisions`, then evaluation and single-question
inference. This is a **correctness** check, not a calibration result.

Setup (once, from a terminal in the repo root):

```bash
uv sync          # add `--extra rich` for the pretty training summary
uv run jupyter lab
```


In [ ]:
from pathlib import Path
import os, sys

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)


In [ ]:
import torch, transformers, datasets
from src.utils.model_utils import detect_device

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("device:", detect_device())


## 1. Train one epoch with the tiny encoder

`configs/rlcd_smoke.yaml` uses `google/bert_uncased_L-2_H-128_A-2` (~4.4M params),
a small batch, and one epoch, so this finishes in under a minute on CPU. It also
needs network access to the Hugging Face Hub the first time (dataset + encoder).


In [ ]:
from src.pipelines.config import load_training_config
from src.pipelines.train import train

cfg = load_training_config(REPO_ROOT / "configs" / "rlcd_smoke.yaml")
history, final = train(cfg)


## 2. Inspect the final test metrics


In [ ]:
import json
print("raw:", json.dumps(final["raw"], indent=2))
print("post-temperature:", json.dumps(final["post_temperature"], indent=2))
print("fitted T:", final["fitted_temperature"])


## 3. Score one typed question


In [ ]:
from src.pipelines.infer import infer

ckpt = Path(cfg.ckpt_dir) / (cfg.run_name or "rlcd_smoke") / "best.pt"
state = "The customer was billed twice for March and asks for a refund today."
question = {
    "type": "choice",
    "instructions": "Which department should handle this request?",
    "criteria": {
        "billing": "invoices, payments, refunds",
        "technical": "bugs, outages, system errors",
    },
}
infer(ckpt, state, question)


The pipeline works end to end. For a real run with a pretrained encoder see
`02_train.ipynb`; for a checkpoint walkthrough see `03_evaluate_and_infer.ipynb`.
